# 02 — Upload Raw CSVs to S3 (Bronze Layer)

**Goal:** Push original Land Registry CSV files from local disk to the AWS S3 bucket's `raw/csv/` location.

This is the **Bronze layer** of the data lake — raw, untouched, immutable. We never modify these files. They represent the source of truth from HM Land Registry.

**Why upload raw CSVs at all?**
- Mirrors production patterns: source data lands in lake before any processing
- Enables full re-processing from scratch if logic changes
- Cheap storage (~£0.02/month for 600MB)
- Auditable: anyone can trace any transformation back to the original

**Path:** `data/raw/pp-*.csv` → `s3://london-housing-analytics-irfan/raw/csv/`

## Imports

In [1]:
import os
import boto3
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load_dotenv(PROJECT_ROOT / '.env')

required = ['AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_REGION', 'S3_BUCKET']
missing = [v for v in required if not os.getenv(v)]
if missing:
    raise ValueError(f"Missing env vars: {missing}")

print(f"✅ AWS credentials loaded")
print(f"Region: {os.getenv('AWS_REGION')}")
print(f"Bucket: {os.getenv('S3_BUCKET')}")

✅ AWS credentials loaded
Region: us-west-2
Bucket: london-housing-analytics-irfan


## Connect to S3

In [5]:
s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
    region_name=os.getenv('AWS_REGION')
)

bucket = os.getenv('S3_BUCKET')

response = s3.list_objects_v2(Bucket=bucket, MaxKeys=5)

print(f" Connected to S3")
print(f"   Bucket: {bucket}")
print(f"   Files currently in bucket: {response.get('KeyCount', 0)}")

 Connected to S3
   Bucket: london-housing-analytics-irfan
   Files currently in bucket: 4


## Upload All CSVs

In [7]:
raw_dir = PROJECT_ROOT / 'data' / 'raw'
bucket = os.getenv('S3_BUCKET')

csv_files = sorted(raw_dir.glob('pp-*.csv'))
print(f"Found {len(csv_files)} CSV files to upload\n")

for csv_file in csv_files:
    s3_key = f'raw/csv/{csv_file.name}'
    size_mb = csv_file.stat().st_size / (1024 * 1024)
    
    print(f"📤 Uploading {csv_file.name} ({size_mb:.1f} MB)...")
    s3.upload_file(
        Filename=str(csv_file),
        Bucket=bucket,
        Key=s3_key
    )
    print(f"    → s3://{bucket}/{s3_key}")

print(f"\n All {len(csv_files)} files uploaded to Bronze layer")

Found 4 CSV files to upload

📤 Uploading pp-2021.csv (212.8 MB)...
    → s3://london-housing-analytics-irfan/raw/csv/pp-2021.csv
📤 Uploading pp-2022.csv (179.0 MB)...
    → s3://london-housing-analytics-irfan/raw/csv/pp-2022.csv
📤 Uploading pp-2023.csv (143.1 MB)...
    → s3://london-housing-analytics-irfan/raw/csv/pp-2023.csv
📤 Uploading pp-2024.csv (154.0 MB)...
    → s3://london-housing-analytics-irfan/raw/csv/pp-2024.csv

 All 4 files uploaded to Bronze layer


## Verify

In [8]:
response = s3.list_objects_v2(Bucket=bucket, Prefix='raw/csv/')

print(f"Files in s3://{bucket}/raw/csv/ (Bronze layer):\n")
total_mb = 0
for obj in response.get('Contents', []):
    size_mb = obj['Size'] / (1024 * 1024)
    total_mb += size_mb
    print(f"  - {obj['Key']} ({size_mb:.1f} MB)")

print(f"\nTotal Bronze layer size: {total_mb:.1f} MB")

Files in s3://london-housing-analytics-irfan/raw/csv/ (Bronze layer):

  - raw/csv/ (0.0 MB)
  - raw/csv/pp-2021.csv (212.8 MB)
  - raw/csv/pp-2022.csv (179.0 MB)
  - raw/csv/pp-2023.csv (143.1 MB)
  - raw/csv/pp-2024.csv (154.0 MB)

Total Bronze layer size: 688.9 MB
